In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [37]:
train_df = pd.read_csv(
    "/kaggle/input/cred-resolve-intelligence-challenge-the-next-best-selection/train.csv"
)
test_df = pd.read_csv(
    "/kaggle/input/cred-resolve-intelligence-challenge-the-next-best-selection/test.csv"
)
meta_df = pd.read_csv(
    "/kaggle/input/cred-resolve-intelligence-challenge-the-next-best-selection/metaData.csv"
)
print(train_df.shape, test_df.shape, meta_df.shape)

(80000, 4) (20000, 3) (100000, 4)


In [38]:
train_df = train_df.merge(meta_df, on="lead_code", how="left")
test_df  = test_df.merge(meta_df, on="lead_code", how="left")
print(train_df.shape, test_df.shape)

(80000, 7) (20000, 6)


In [39]:
X = train_df.drop(columns=["TARGET", "id"])
y = train_df["TARGET"]
X_test = test_df.drop(columns=["id"])

In [40]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()
print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['lead_code', 'suggested_action', 'dpd_bucket', 'state']
Numeric: ['total_due']


In [41]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols)
    ]
)
model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("regressor", LinearRegression())
    ]
)

In [42]:
model.fit(X, y)
print("Model training completed ")

Model training completed 


In [43]:
preds = model.predict(X_test)
preds = np.clip(preds, 0, 1)
preds[:10]

array([0.4489238 , 0.44893577, 0.44893894, 0.44893365, 0.44891348,
       0.44896078, 0.44892245, 0.44894324, 0.44892478, 0.44892034])

In [44]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "TARGET": preds
})
submission.to_csv("submission.csv", index=False)
submission.head()

,id,TARGET
0,80001,0.448924
1,80002,0.448936
2,80003,0.448939
3,80004,0.448934
4,80005,0.448913
